# 🧩 vLLM 0.32 × モデル/設定の互換性（install 不要）

依存ピン・arch 対応・reasoning parser 名・kv-cache 値域・awq 条件。


In [ ]:
# vLLM 0.32 wheel × このモデル/設定 の互換性を静的確認（install 不要）
import glob
import json
import os
import pathlib
import re
import subprocess
import sys
import zipfile

ROOT = ("/kaggle/input/datasets/yamahei21python/"
        "qwen38-27b-uncensored-hauhaucs-awq-mtp")
cfg = json.load(open(os.path.join(ROOT, "config.json")))
ARCH = (cfg.get("architectures") or ["<none>"])[0]
MODEL_TYPE = cfg.get("model_type")
print("architectures:", ARCH, "| model_type:", MODEL_TYPE)

r = subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                    "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
                   capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("download rc:", r.returncode)
if not whl:
    raise SystemExit("wheel 取得失敗")
z = zipfile.ZipFile(whl[0])
z.extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")

# --- 1. 依存ピン（transformers / torch の要求）---
meta_name = [n for n in z.namelist() if n.endswith(".dist-info/METADATA")][0]
meta = z.read(meta_name).decode("utf-8", "replace")
print("\n=== Requires-Dist（主要のみ）===")
for line in meta.splitlines():
    if line.startswith("Requires-Dist:"):
        low = line.lower()
        if any(k in low for k in ("torch", "transformers", "tokenizers",
                                  "safetensors", "numpy", "huggingface")):
            print("  ", line[len("Requires-Dist: "):][:140])
print("Requires-Python:", [l for l in meta.splitlines()
                           if l.startswith("Requires-Python")])

# --- 2. モデルアーキ対応 ---
reg = root / "model_executor/models/registry.py"
txt = reg.read_text(errors="replace") if reg.exists() else ""
print("\n=== arch 対応 ===")
print("registry exists:", reg.exists())
print("arch in registry:", ARCH in txt)
for m in re.finditer(r'"([A-Za-z0-9_]+ForCausalLM)"\s*:\s*"([^"]+)"', txt):
    if "Qwen" in m.group(1):
        print("  ", m.group(1), "->", m.group(2))

# --- 3. reasoning parser 名（--reasoning-parser qwen3 が有効か）---
print("\n=== reasoning parser 名 ===")
rp = list(root.rglob("reasoning/*.py")) + list(root.rglob("reasoning*.py"))
names = set()
for f in rp:
    t = f.read_text(errors="replace")
    for m in re.finditer(r'register_reasoning_parser\(["\']([^"\']+)', t):
        names.add(m.group(1))
print("registered:", sorted(names))
print("qwen3 available:", "qwen3" in names)
for f in rp:
    if "qwen3" in f.name:
        print("  file:", f.relative_to(root))

# --- 4. kv-cache-dtype の取り得る値 ---
print("\n=== kv-cache-dtype choices ===")
cache = (root / "config/cache.py").read_text(errors="replace")
m = re.search(r"CacheDType\s*=\s*Literal\[(.*?)\]", cache, re.S)
print(m.group(0)[:400] if m else "not found")

# --- 5. speculative の mpt 検証（method=mtp に model が要るか）---
print("\n=== speculative: mtp 分岐 ===")
sp = (root / "config/speculative.py").read_text(errors="replace")
for i, line in enumerate(sp.splitlines(), 1):
    s = line.strip()
    if "mtp" in s.lower() and ("method" in s or "if " in s or "assert" in s
                               or "raise" in s):
        print(f"  {i}: {s[:170]}")

# --- 6. 量子化 awq のサポート条件（group_size 等）---
print("\n=== awq 条件 ===")
for f in root.rglob("quantization/awq*.py"):
    t = f.read_text(errors="replace")
    for i, line in enumerate(t.splitlines(), 1):
        s = line.strip()
        if ("group_size" in s or "bits" in s or "zero_point" in s) and (
                "if " in s or "raise" in s or "==" in s):
            print(f"  {f.name}:{i}: {s[:150]}")
